
# Phase 3: Predictor Availability and Leakage Audit

## Objective

The objective of this phase is to determine which variables can be used to forecast industrial energy consumption at 15-minute and 1-hour horizons.

The audit distinguishes between:
- Variables known at the forecast origin
- Historical measurements that can be used as lagged predictors
- Variables whose availability or derivation requires further investigation
- Future target variables that must never be used as predictors

The purpose is to establish a defensible, leakage-aware feature-selection policy before model training.


In [1]:

import pandas as pd
from pathlib import Path

INPUT_PATH = Path("../Data/processed/steel_industry_phase2_targets.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Phase 2 dataset not found: {INPUT_PATH.resolve()}"
    )

df = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])

required_columns = [
    "timestamp",
    "Usage_kWh",
    "target_15min",
    "target_1hour",
    "NSM",
    "WeekStatus",
    "Day_of_week",
    "Load_Type",
    "CO2(tCO2)",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor"
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

df = df.sort_values("timestamp").reset_index(drop=True)

assert df["timestamp"].notna().all(), "Invalid timestamps found."
assert df["timestamp"].is_unique, "Duplicate timestamps found."
assert df["timestamp"].is_monotonic_increasing, "Timestamps are not sorted."

print("Phase 2 dataset loaded.")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Period: {df['timestamp'].min()} to {df['timestamp'].max()}")

display(df.head())


Phase 2 dataset loaded.
Rows: 35,040
Columns: 16
Period: 2018-01-01 00:00:00 to 2018-12-31 23:45:00


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,timestamp,zero_usage_flag,high_usage_flag,target_15min,target_1hour
0,01-01-2018 00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load,2018-01-01 00:00:00,False,False,3.17,3.31
1,01-01-2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load,2018-01-01 00:15:00,False,False,4.00,3.82
2,01-01-2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load,2018-01-01 00:30:00,False,False,3.24,3.28
3,01-01-2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load,2018-01-01 00:45:00,False,False,3.31,3.60
4,01-01-2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load,2018-01-01 01:00:00,False,False,3.82,3.60



## 1. Predictor Availability Policy

A variable is eligible as a predictor only if its value would be available when the forecast is issued.

Historical energy and sensor measurements may be used as lagged features. Calendar features may be derived from the timestamp. Future target columns are strictly excluded.

The availability of contemporaneous sensor readings, CO2 measurements, and load-type labels must be established from the data-generation process or operational documentation before they are approved.


In [2]:

audit = pd.DataFrame([
    {
        "variable": "timestamp",
        "role": "Time reference",
        "initial_policy": "Derive calendar features",
        "status": "Eligible"
    },
    {
        "variable": "Usage_kWh",
        "role": "Energy consumption",
        "initial_policy": "Use historical/current values and lags",
        "status": "Eligible subject to forecast-origin definition"
    },
    {
        "variable": "NSM",
        "role": "Seconds from midnight",
        "initial_policy": "Verify against timestamp",
        "status": "Validate"
    },
    {
        "variable": "WeekStatus",
        "role": "Weekday/weekend label",
        "initial_policy": "Verify against timestamp",
        "status": "Validate"
    },
    {
        "variable": "Day_of_week",
        "role": "Day label",
        "initial_policy": "Verify against timestamp",
        "status": "Validate"
    },
    {
        "variable": "Lagging_Current_Reactive.Power_kVarh",
        "role": "Electrical sensor",
        "initial_policy": "Use historical lags initially",
        "status": "Validate measurement timing"
    },
    {
        "variable": "Leading_Current_Reactive_Power_kVarh",
        "role": "Electrical sensor",
        "initial_policy": "Use historical lags initially",
        "status": "Validate measurement timing"
    },
    {
        "variable": "Lagging_Current_Power_Factor",
        "role": "Electrical sensor",
        "initial_policy": "Use historical lags initially",
        "status": "Validate measurement timing"
    },
    {
        "variable": "Leading_Current_Power_Factor",
        "role": "Electrical sensor",
        "initial_policy": "Use historical lags initially",
        "status": "Validate measurement timing"
    },
    {
        "variable": "CO2(tCO2)",
        "role": "Emissions measurement",
        "initial_policy": "Do not approve until timing and derivation are known",
        "status": "Investigate"
    },
    {
        "variable": "Load_Type",
        "role": "Operating-condition label",
        "initial_policy": "Check how and when the label is assigned",
        "status": "Investigate"
    },
    {
        "variable": "target_15min",
        "role": "Future outcome",
        "initial_policy": "Target only",
        "status": "Never use as predictor"
    },
    {
        "variable": "target_1hour",
        "role": "Future outcome",
        "initial_policy": "Target only",
        "status": "Never use as predictor"
    }
])

display(audit)


,variable,role,initial_policy,status
0,timestamp,Time reference,Derive calendar features,Eligible
1,Usage_kWh,Energy consumption,Use historical/current values and lags,Eligible subject to forecast-origin definition
2,NSM,Seconds from midnight,Verify against timestamp,Validate
3,WeekStatus,Weekday/weekend label,Verify against timestamp,Validate
4,Day_of_week,Day label,Verify against timestamp,Validate
5,Lagging_Current_Reactive.Power_kVarh,Electrical sensor,Use historical lags initially,Validate measurement timing
6,Leading_Current_Reactive_Power_kVarh,Electrical sensor,Use historical lags initially,Validate measurement timing
7,Lagging_Current_Power_Factor,Electrical sensor,Use historical lags initially,Validate measurement timing
8,Leading_Current_Power_Factor,Electrical sensor,Use historical lags initially,Validate measurement timing
9,CO2(tCO2),Emissions measurement,Do not approve until timing and derivation are...,Investigate



## 2. Validate Time-Derived Variables

The timestamp is the reference for calendar information. Compare the recorded time-related variables with values derived from the timestamp.

Mismatches should be investigated before using these variables in the forecasting models.


In [3]:

# Derive expected values from timestamp
expected_nsm = (
    df["timestamp"].dt.hour * 3600
    + df["timestamp"].dt.minute * 60
    + df["timestamp"].dt.second
)

expected_day = df["timestamp"].dt.day_name()

expected_weekstatus = pd.Series(
    "Weekday",
    index=df.index
)
expected_weekstatus.loc[df["timestamp"].dt.dayofweek >= 5] = "Weekend"

# Normalize recorded categorical labels for comparison
recorded_day = df["Day_of_week"].astype("string").str.strip()
recorded_weekstatus = df["WeekStatus"].astype("string").str.strip()

print("NSM mismatches:",
      int((pd.to_numeric(df["NSM"], errors="coerce") != expected_nsm).sum()))

print("Day-of-week mismatches:",
      int((recorded_day != expected_day).sum()))

print("WeekStatus mismatches:",
      int((recorded_weekstatus != expected_weekstatus).sum()))

# Display examples for investigation
day_mismatch = recorded_day != expected_day
week_mismatch = recorded_weekstatus != expected_weekstatus

if day_mismatch.any():
    print("\nExamples of day-of-week mismatches:")
    display(df.loc[day_mismatch, ["timestamp", "Day_of_week"]].head(10))

if week_mismatch.any():
    print("\nExamples of WeekStatus mismatches:")
    display(df.loc[week_mismatch, ["timestamp", "WeekStatus"]].head(10))


NSM mismatches: 0
Day-of-week mismatches: 0
WeekStatus mismatches: 0



## 3. Investigate CO2 and Load_Type

CO2 measurements and load-type labels may contain useful operational information, but their predictive validity depends on when they become available.

This section summarizes their distributions and associations with energy consumption. These descriptive results do not, by themselves, establish that either variable is safe for forecasting.


In [4]:

print("CO2 summary statistics:")
display(df["CO2(tCO2)"].describe().to_frame())

print("Unique CO2 values:", df["CO2(tCO2)"].nunique())

print("\nCorrelation between current energy usage and CO2:")
display(
    df[["Usage_kWh", "CO2(tCO2)"]]
    .corr(numeric_only=True)
)

print(
    "\nDecision: CO2(tCO2) remains under investigation until "
    "its measurement timing and derivation are documented."
)


CO2 summary statistics:


,CO2(tCO2)
count,35040.000000
mean,0.011524
std,0.016151
min,0.000000
25%,0.000000
50%,0.000000
75%,0.020000
max,0.070000


Unique CO2 values: 8

Correlation between current energy usage and CO2:


,Usage_kWh,CO2(tCO2)
Usage_kWh,1.00000,0.98818
CO2(tCO2),0.98818,1.00000



Decision: CO2(tCO2) remains under investigation until its measurement timing and derivation are documented.


In [8]:
# Compare CO2 values for different energy-consumption readings
display(
    df.groupby("Usage_kWh")["CO2(tCO2)"]
      .agg(["count", "mean", "min", "max"])
      .sort_index()
      .head(20)
)

# Check the ratio for observations with positive energy use
positive_usage = df[df["Usage_kWh"] > 0].copy()

positive_usage["CO2_per_kWh"] = (
    positive_usage["CO2(tCO2)"] / positive_usage["Usage_kWh"]
)

display(positive_usage["CO2_per_kWh"].describe())

,count,mean,min,max
Usage_kWh,,,,
0.00,1,0.0,0.0,0.0
2.45,8,0.0,0.0,0.0
2.48,36,0.0,0.0,0.0
2.52,110,0.0,0.0,0.0
2.56,157,0.0,0.0,0.0
2.59,223,0.0,0.0,0.0
2.63,240,0.0,0.0,0.0
2.66,305,0.0,0.0,0.0
2.70,328,0.0,0.0,0.0


count    35039.000000
mean         0.000185
std          0.000231
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000440
max          0.000917
Name: CO2_per_kWh, dtype: float64

In [5]:

print("Load_Type frequencies:")
display(
    df["Load_Type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nEnergy consumption by Load_Type:")
display(
    df.groupby("Load_Type", dropna=False)["Usage_kWh"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
)

print(
    "\nDecision: Load_Type remains under investigation until "
    "its assignment rule and availability at forecast time are confirmed."
)


Load_Type frequencies:


,count
Load_Type,
Light_Load,18072
Medium_Load,9696
Maximum_Load,7272



Energy consumption by Load_Type:


,count,mean,median,min,max
Load_Type,,,,,
Light_Load,18072,8.626,3.310,0.00,140.29
Maximum_Load,7272,59.265,56.630,2.92,151.67
Medium_Load,9696,38.445,34.435,2.52,157.18



Decision: Load_Type remains under investigation until its assignment rule and availability at forecast time are confirmed.


In [9]:

# Check whether Load_Type changes across the day
df["hour"] = df["timestamp"].dt.hour

display(
    df.groupby(["hour", "Load_Type"])
      .size()
      .unstack(fill_value=0)
)

# Check whether Load_Type is determined by current energy consumption
display(
    df.groupby("Load_Type")["Usage_kWh"]
      .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
      .unstack()
      .round(3)
)


Load_Type,Light_Load,Maximum_Load,Medium_Load
hour,,,
0,1460,0,0
1,1460,0,0
2,1460,0,0
3,1460,0,0
4,1460,0,0
5,1460,0,0
6,1460,0,0
7,1460,0,0
8,1460,0,0


,0.10,0.25,0.50,0.75,0.90
Load_Type,,,,,
Light_Load,2.74,2.950,3.310,4.03,7.236
Maximum_Load,21.53,42.002,56.630,75.31,102.917
Medium_Load,3.56,4.420,34.435,61.70,91.245



## 4. Final Audit Decisions

The notebook records the current predictor policy and outstanding questions.

Calendar variables may be derived from the timestamp after validation. Historical energy and sensor readings may be considered as lagged features. CO2 and Load_Type require further evidence before inclusion.

Future target variables must never enter the predictor matrix. The next phase will implement feature engineering using the approved policy.


In [7]:

OUTPUT_DIR = Path("../Data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_path = OUTPUT_DIR / "phase3_predictor_availability_audit.csv"
audit.to_csv(audit_path, index=False)

print("Predictor availability audit saved:")
print(audit_path.resolve())


Predictor availability audit saved:
D:\Research Paper Codes\Data\processed\phase3_predictor_availability_audit.csv



## Predictor Availability Audit — Preliminary Decisions

| Variable | Decision | Reason |
|---|---|---|
| NSM | Eligible | Matches timestamp |
| Day_of_week | Eligible | Matches timestamp |
| WeekStatus | Eligible | Matches timestamp |
| Usage_kWh | Eligible as historical lags | Must be available at forecast origin |
| Electrical sensor variables | Historical lags only initially | Verify measurement timing |
| CO2(tCO2) | Exclude initially | Derivation and timing unverified |
| Load_Type | Exclude initially | Assignment rule and availability unverified |
| target_15min | Never a predictor | Forecast target |
| target_1hour | Never a predictor | Forecast target |

These are preliminary decisions. Final feature inclusion must follow the forecast-origin definition and leakage-safe validation.
